# Model 1: Qwen2.5-3B Fine-tuned on SQuAD v2
## Baseline QA with ROUGE & Hallucination Evaluation

### Goals
1. Fine-tune `Qwen2.5-3B-Instruct` on [SQuAD v2](https://huggingface.co/datasets/squad_v2)
2. Generate answers on the validation set
3. Evaluate using:
   - **ROUGE-1, ROUGE-2, ROUGE-L** (lexical overlap)
   - **Faithfulness / Hallucination Score** (NLI-based)

### Architecture
```
SQuAD v2 Dataset → Qwen2.5 Chat Prompt → QLoRA Fine-tune → Inference → ROUGE + Faithfulness
```
### Why SQuAD v2 for fine-tuning?
- Clean Wikipedia-grounded QA pairs with gold reference answers
- Has unanswerable questions — great for hallucination testing
- The model learns QA format WITHOUT memorizing TriviaQA (used as RAG KB)
- This separation ensures the RAG comparison is fair and meaningful

In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"
print("✅ CUDA debug flags set")

In [ ]:
import torch
import transformers, bitsandbytes, peft, trl

print("📦 Package versions:")
print(f"   torch          : {torch.__version__}")
print(f"   transformers   : {transformers.__version__}")
print(f"   bitsandbytes   : {bitsandbytes.__version__}")
print(f"   peft           : {peft.__version__}")
print(f"   trl            : {trl.__version__}")
print(f"   CUDA           : {torch.version.cuda}")
print()
print("🖥️  GPU:")
if torch.cuda.is_available():
    print(f"   Device : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"   VRAM used (should be ~0): {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("   ❌ No GPU found")

import bitsandbytes as bnb
_ = bnb.nn.Linear4bit(16, 16)
print()
print("✅ All checks passed!")

In [ ]:
import os, gc, json, torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime

from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
)
from trl import SFTTrainer, SFTConfig
from rouge_score import rouge_scorer
from sentence_transformers import CrossEncoder

# ── Config ─────────────────────────────────────────────────────────────────────
MODEL_NAME    = "Qwen/Qwen2.5-3B-Instruct"
OUTPUT_DIR    = "/content/qwen25-squadv2"
MERGED_DIR    = "/content/qwen25-squadv2-merged"
SAVE_DIR      = "./model1_outputs"
MAX_SEQ_LEN   = 512
TRAIN_SAMPLES = 5000
EVAL_SAMPLES  = 200
BATCH_SIZE    = 4
GRAD_ACCUM    = 4
EPOCHS        = 2
LR            = 1e-4
LORA_R        = 8
LORA_ALPHA    = 16
LORA_DROPOUT  = 0.05
SEED          = 42

torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"✅ Config ready")
print(f"   Model  : {MODEL_NAME}")
print(f"   Device : {device} — {torch.cuda.get_device_name(0)}")
print(f"   VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB total")
print(f"   Fine-tune dataset : SQuAD v2 ({TRAIN_SAMPLES:,} samples)")
print(f"   Eval dataset      : SQuAD v2 validation ({EVAL_SAMPLES:,} samples)")

## Part 2: Load & Explore SQuAD v2

In [ ]:
print("📥 Loading SQuAD v2...")
dataset = load_dataset("squad_v2")

for split, d in dataset.items():
    print(f"   {split:12s}: {len(d):,} samples")

ex = dataset['train'][0]
print(f"\n🔍 Sample question : {ex['question']}")
print(f"   Context (first 200): {ex['context'][:200]}...")
print(f"   Answer : {ex['answers']['text'][0] if ex['answers']['text'] else '[unanswerable]'}")
print()
print("ℹ️  SQuAD v2 includes unanswerable questions — good for hallucination testing!")

In [ ]:
SYSTEM = (
    "You are a helpful and precise question-answering assistant. "
    "Answer the question based on the provided context. "
    "If the context does not contain enough information to answer, say 'I cannot determine the answer from the given context.'"
)

def format_sample(sample):
    """Full prompt + answer for training (Qwen2.5 ChatML format)."""
    context  = sample['context']
    question = sample['question']
    answers  = sample['answers']['text']

    if answers:
        answer = answers[0]
    else:
        answer = "I cannot determine the answer from the given context."

    user = f"Context:\n{context}\n\nQuestion: {question}"
    text = (
        f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )
    return {"text": text, "answer": answer, "question": question, "context": context}

def inference_prompt(sample):
    """Prompt only — no assistant turn — for inference."""
    context  = sample['context']
    question = sample['question']
    user     = f"Context:\n{context}\n\nQuestion: {question}"
    return (
        f"<|im_start|>system\n{SYSTEM}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def ref_answer(sample):
    answers = sample['answers']['text']
    return answers[0] if answers else "I cannot determine the answer from the given context."

# Filter out samples where context is empty, then format
train_raw  = dataset['train'].filter(lambda x: len(x['context']) > 50)
val_raw    = dataset['validation'].filter(lambda x: len(x['context']) > 50)

train_data = train_raw.select(range(TRAIN_SAMPLES)).map(format_sample)
val_data   = val_raw.select(range(EVAL_SAMPLES)).map(format_sample)

print(f"✅ Formatted: {len(train_data):,} train | {len(val_data):,} val")
print(f"\n📝 Example (truncated):")
print(train_data[0]['text'][:500])

In [ ]:
import gc, torch

for name in ['model', 'base_model', 'trainer', 'gen_pipe']:
    if name in dir():
        exec(f'del {name}')

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(f"VRAM free  : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"VRAM total : {torch.cuda.mem_get_info()[1]/1e9:.2f} GB")

In [ ]:
if 'model' in dir():
    del model
gc.collect()
torch.cuda.empty_cache()

print(f"VRAM free before loading: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

print("📥 Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
print(f"   Vocab size : {tokenizer.vocab_size:,}")

print(f"\n📥 Loading {MODEL_NAME} in float16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True,
    attn_implementation="eager",
)
model.config.use_cache = False
model.config.pretraining_tp = 1
print(f"   ✅ Base model loaded")
print(f"   VRAM used : {torch.cuda.memory_allocated()/1e9:.2f} GB")

model.enable_input_require_grads()
model.gradient_checkpointing_enable()
print("   ✅ Gradient checkpointing enabled")

lora_cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none",
)
model = get_peft_model(model, lora_cfg)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())

print(f"\n📊 Parameters:")
print(f"   Trainable : {trainable:,}  ({100*trainable/total:.2f}%)")
print(f"   Frozen    : {total-trainable:,}")
print(f"   Total     : {total:,}")
print("\n✅ Model ready for training!")

## Fine-Tune with SFTTrainer

In [ ]:
import shutil

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
    print(f"🗑️  Deleted old checkpoint dir: {OUTPUT_DIR}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Clean output dir ready")

In [ ]:
checkpoint = None
if os.path.exists(OUTPUT_DIR):
    checkpoints = [
        os.path.join(OUTPUT_DIR, d)
        for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-")
    ]
    if checkpoints:
        checkpoint = max(checkpoints, key=os.path.getmtime)
        print(f"📂 Resuming from checkpoint: {checkpoint}")
    else:
        print("🆕 No checkpoint found — starting fresh")

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    bf16=False,
    fp16=True,
    logging_steps=25,
    save_steps=50,
    save_total_limit=3,
    max_grad_norm=0.3,
    weight_decay=0.001,
    group_by_length=True,
    report_to="none",
    seed=SEED,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=sft_config,
)

print("🚀 Starting fine-tuning...")
print(f"   {TRAIN_SAMPLES:,} samples × {EPOCHS} epochs")
print(f"   Effective batch : {BATCH_SIZE} × {GRAD_ACCUM} = {BATCH_SIZE*GRAD_ACCUM}")
print(f"   Steps total     : {(TRAIN_SAMPLES // (BATCH_SIZE*GRAD_ACCUM)) * EPOCHS}")

t0 = datetime.now()
trainer.train(resume_from_checkpoint=checkpoint)
elapsed = datetime.now() - t0
print(f"\n✅ Training done in {elapsed}")

print("\n💾 Saving adapters...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"   Adapters saved → {os.path.abspath(OUTPUT_DIR)}")

del trainer
gc.collect()
torch.cuda.empty_cache()
print(f"   VRAM free after clearing trainer: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

os.makedirs(MERGED_DIR, exist_ok=True)
print(f"\n💾 Saving merged model → {os.path.abspath(MERGED_DIR)}...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"✅ Merged model saved — RAG notebook will load from this path")

## Inference

In [ ]:
model.config.use_cache = True
model.eval()

gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free : {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")
print(f"Model type: {type(model).__name__}")

def generate_answer(sample, max_new_tokens=150):
    prompt = inference_prompt(sample)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.1,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response   = tokenizer.decode(new_tokens, skip_special_tokens=False)
    return response.replace("<|im_end|>", "").strip()

print(f"\n🔮 Generating answers for {EVAL_SAMPLES} samples...")

results    = []
val_subset = val_raw.select(range(EVAL_SAMPLES))

t0 = datetime.now()
for i, sample in enumerate(tqdm(val_subset, desc="Inference")):
    results.append({
        "id":        i,
        "question":  sample['question'],
        "context":   sample['context'],
        "generated": generate_answer(sample),
        "reference": ref_answer(sample),
        "answerable": len(sample['answers']['text']) > 0,
    })

results_df = pd.DataFrame(results)
print(f"\n✅ Done in {datetime.now()-t0} — {len(results_df)} answers generated")
print(f"\nSample:")
print(f"Q        : {results_df.iloc[0]['question']}")
print(f"Reference: {results_df.iloc[0]['reference']}")
print(f"Generated: {results_df.iloc[0]['generated'][:200]}")

## ROUGE Evaluation

In [ ]:
import re

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
r1, r2, rL = [], [], []
exact_matches = []

def normalise(text):
    text = text.lower().strip()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text

for _, row in tqdm(results_df.iterrows(), total=len(results_df), desc="ROUGE"):
    gen = row['generated'].strip() or " "
    ref = row['reference']
    s   = scorer.score(ref, gen)
    r1.append(s['rouge1'].fmeasure)
    r2.append(s['rouge2'].fmeasure)
    rL.append(s['rougeL'].fmeasure)
    exact_matches.append(int(normalise(gen) == normalise(ref)))

results_df['rouge1']       = r1
results_df['rouge2']       = r2
results_df['rougeL']       = rL
results_df['exact_match']  = exact_matches

print(f"\n{'='*52}")
print(f"  📊 EVALUATION — Model 1 (Qwen2.5-3B, No RAG)")
print(f"{'='*52}")
print(f"  ROUGE-1     : {np.mean(r1):.4f}  ±{np.std(r1):.4f}")
print(f"  ROUGE-2     : {np.mean(r2):.4f}  ±{np.std(r2):.4f}")
print(f"  ROUGE-L     : {np.mean(rL):.4f}  ±{np.std(rL):.4f}")
print(f"  Exact Match : {np.mean(exact_matches)*100:.1f}%")
print(f"{'='*52}")

In [ ]:
pip install bert-score

In [ ]:
# ── Cell: BERTScore Evaluation ─────────────────────────────────────────────────────────────────────
# pip install bert-score (if not already installed)
# !pip install bert-score -q

from bert_score import score as bert_score_fn

print("🔮 Computing BERTScore (semantic similarity vs gold answers)...")
print("   Using model: microsoft/deberta-xlarge-mnli (best for QA)")

# Extract lists
bs_candidates = results_df['generated'].apply(lambda x: x.strip() or " ").tolist()
bs_references = results_df['reference'].tolist()

# Compute BERTScore — lang='en' auto-selects the best model
P, R, F1 = bert_score_fn(
    bs_candidates,
    bs_references,
    lang="en",
    model_type="microsoft/deberta-xlarge-mnli",
    batch_size=16,
    device=device,
    verbose=True,
)

results_df['bertscore_P']  = P.numpy()
results_df['bertscore_R']  = R.numpy()
results_df['bertscore_F1'] = F1.numpy()

mean_bs_P  = P.mean().item()
mean_bs_R  = R.mean().item()
mean_bs_F1 = F1.mean().item()

print(f"\n{'='*56}")
print(f"  📊 BERTScore — Model 1 (Qwen2.5-3B, No RAG)")
print(f"{'='*56}")
print(f"  Precision : {mean_bs_P:.4f}  ±{P.std().item():.4f}")
print(f"  Recall    : {mean_bs_R:.4f}  ±{R.std().item():.4f}")
print(f"  F1        : {mean_bs_F1:.4f}  ±{F1.std().item():.4f}")
print(f"{'='*56}")
print(f"  💡 BERTScore captures semantic similarity beyond exact n-gram overlap.")


## Faithfulness / Hallucination Evaluation

In [ ]:
gc.collect()
torch.cuda.empty_cache()
print(f"VRAM free: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

print("🧠 Loading NLI model...")
nli = CrossEncoder("cross-encoder/nli-deberta-v3-small", device=device, max_length=512)
print("✅ NLI model loaded")

def faithfulness_batch(premises, hypotheses, bs=32):
    scores = []
    for i in range(0, len(premises), bs):
        pairs  = list(zip(premises[i:i+bs], hypotheses[i:i+bs]))
        logits = nli.predict(pairs, apply_softmax=True)
        scores.extend(logits[:, 1].tolist())
    return scores

# Faithfulness vs context (the passage the answer should be grounded in)
faith_scores = faithfulness_batch(
    results_df['context'].tolist(),
    results_df['generated'].tolist()
)
results_df['faithfulness'] = faith_scores
results_df['hallucinated'] = results_df['faithfulness'] < 0.3

mean_f = np.mean(faith_scores)
halluc = results_df['hallucinated'].mean()

print(f"\n{'='*50}")
print(f"  🧠 FAITHFULNESS — Model 1 (No RAG)")
print(f"{'='*50}")
print(f"  Mean Faithfulness : {mean_f:.4f}")
print(f"  Faithful  (≥0.3)  : {(1-halluc)*100:.1f}%")
print(f"  Hallucinated(<0.3): {halluc*100:.1f}%")
print(f"{'='*50}")

# Answerable vs unanswerable breakdown
ans_df   = results_df[results_df['answerable'] == True]
unans_df = results_df[results_df['answerable'] == False]
print(f"\n  Answerable questions   : {len(ans_df)} | halluc: {ans_df['hallucinated'].mean()*100:.1f}%")
print(f"  Unanswerable questions : {len(unans_df)} | halluc: {unans_df['hallucinated'].mean()*100:.1f}%")
print(f"  (Unanswerable halluc rate should be high for baseline — model guesses instead of abstaining)")

## Visualisation

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Model 1: Qwen2.5-3B Fine-tuned on SQuAD v2 (No RAG) — Full Evaluation',
             fontsize=15, fontweight='bold')

# ROUGE bars
ax = axes[0, 0]
means = [np.mean(r1), np.mean(r2), np.mean(rL)]
stds  = [np.std(r1),  np.std(r2),  np.std(rL)]
bars  = ax.bar(['ROUGE-1','ROUGE-2','ROUGE-L'], means, yerr=stds,
               color=['#2196F3','#4CAF50','#FF9800'], capsize=5, edgecolor='black', alpha=0.85)
ax.set_ylim(0, 1); ax.set_title('ROUGE Scores', fontweight='bold'); ax.set_ylabel('F1 Score')
ax.axhline(0.4, color='red', linestyle='--', alpha=0.5, label='Good threshold')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars, means):
    ax.text(b.get_x()+b.get_width()/2, v+0.01, f'{v:.3f}', ha='center', fontweight='bold', fontsize=10)

# Faithfulness histogram
ax = axes[0, 1]
ax.hist(faith_scores, bins=20, color='#9C27B0', edgecolor='black', alpha=0.8)
ax.axvline(0.3,    color='red',  linestyle='--', lw=2, label='Hallucination threshold')
ax.axvline(mean_f, color='blue', linestyle='-',  lw=2, label=f'Mean = {mean_f:.3f}')
ax.set_title('Faithfulness Distribution', fontweight='bold')
ax.set_xlabel('Faithfulness Score'); ax.set_ylabel('Count')
ax.legend(fontsize=8); ax.grid(axis='y', alpha=0.3)

# Faithful vs Hallucinated pie
ax = axes[0, 2]
ax.pie([1-halluc, halluc],
       labels=[f'Faithful\n{(1-halluc)*100:.1f}%', f'Hallucinated\n{halluc*100:.1f}%'],
       colors=['#4CAF50','#F44336'], autopct='%1.1f%%', startangle=90,
       wedgeprops={'edgecolor':'white','linewidth':2})
ax.set_title('Faithful vs Hallucinated', fontweight='bold')

# Answerable vs Unanswerable hallucination
ax = axes[1, 0]
cats = ['Answerable', 'Unanswerable']
halluc_rates = [ans_df['hallucinated'].mean()*100, unans_df['hallucinated'].mean()*100]
bars2 = ax.bar(cats, halluc_rates, color=['#2196F3','#F44336'], edgecolor='black', alpha=0.85, width=0.4)
ax.set_ylim(0, 100); ax.set_title('Hallucination by Answerability', fontweight='bold')
ax.set_ylabel('Hallucination Rate (%)'); ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars2, halluc_rates):
    ax.text(b.get_x()+b.get_width()/2, v+0.8, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=12)

# All metrics
ax = axes[1, 1]
mnames  = ['Exact Match', 'ROUGE-L', 'ROUGE-1', 'Faithful']
mvalues = [np.mean(exact_matches)*100, np.mean(rL)*100, np.mean(r1)*100, (1-halluc)*100]
bars3 = ax.bar(mnames, mvalues, color=['#FF5722','#FF9800','#2196F3','#4CAF50'], edgecolor='black', alpha=0.85)
ax.set_ylim(0, 100); ax.set_title('All Metrics (%)', fontweight='bold')
ax.set_ylabel('Score (%)'); ax.grid(axis='y', alpha=0.3)
for b, v in zip(bars3, mvalues):
    ax.text(b.get_x()+b.get_width()/2, v+0.8, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=10)

# ROUGE per answerable/unanswerable
ax = axes[1, 2]
ans_r1   = results_df[results_df['answerable']==True]['rouge1'].mean()
unans_r1 = results_df[results_df['answerable']==False]['rouge1'].mean()
ax.bar(['Answerable\nROUGE-1', 'Unanswerable\nROUGE-1'], [ans_r1, unans_r1],
       color=['#00BCD4','#FF9800'], edgecolor='black', alpha=0.85)
ax.set_ylim(0, 1); ax.set_title('ROUGE-1 by Answerability', fontweight='bold')
ax.set_ylabel('ROUGE-1 F1'); ax.grid(axis='y', alpha=0.3)
for i, v in enumerate([ans_r1, unans_r1]):
    ax.text(i, v+0.01, f'{v:.3f}', ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
os.makedirs(SAVE_DIR, exist_ok=True)
plt.savefig(f"{SAVE_DIR}/model1_results.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"📊 Saved → {os.path.abspath(SAVE_DIR)}/model1_results.png")

## Save Results & Metrics

In [ ]:
os.makedirs(SAVE_DIR, exist_ok=True)
results_df.to_csv(f"{SAVE_DIR}/model1_results.csv", index=False)

model1_metrics = {
    "model":         "Qwen2.5-3B-Instruct (Fine-tuned on SQuAD v2, No RAG)",
    "train_samples": TRAIN_SAMPLES,
    "eval_samples":  EVAL_SAMPLES,
    "dataset":       "SQuAD v2",
    "rouge": {
        "rouge1":     round(float(np.mean(r1)), 4),
        "rouge2":     round(float(np.mean(r2)), 4),
        "rougeL":     round(float(np.mean(rL)), 4),
        "rouge1_std": round(float(np.std(r1)),  4),
        "rouge2_std": round(float(np.std(r2)),  4),
        "rougeL_std": round(float(np.std(rL)),  4),
    },
    "faithfulness": {
        "mean_score":         round(float(mean_f),     4),
        "faithfulness_rate":  round(float(1-halluc),   4),
        "hallucination_rate": round(float(halluc),     4),
        "samples_evaluated":  int(len(results_df)),
    },
    "bertscore": {
        "precision":     round(mean_bs_P,          4),
        "recall":        round(mean_bs_R,          4),
        "f1":            round(mean_bs_F1,         4),
        "precision_std": round(P.std().item(),     4),
        "recall_std":    round(R.std().item(),     4),
        "f1_std":        round(F1.std().item(),    4),
        "model":         "microsoft/deberta-xlarge-mnli",
    },
    "exact_match": round(float(np.mean(exact_matches)), 4),
}

with open(f"{SAVE_DIR}/model1_metrics.json", 'w') as f:
    json.dump(model1_metrics, f, indent=2)

print(f"\n💾 Saved to: {os.path.abspath(SAVE_DIR)}")
print(f"   model1_results.csv")
print(f"   model1_metrics.json")
print(f"   model1_results.png")
print()
print("=" * 52)
print("  ✅ MODEL 1 COMPLETE — SUMMARY")
print("=" * 52)
print(f"  ROUGE-1     : {model1_metrics['rouge']['rouge1']:.4f}")
print(f"  ROUGE-2     : {model1_metrics['rouge']['rouge2']:.4f}")
print(f"  ROUGE-L     : {model1_metrics['rouge']['rougeL']:.4f}")
print(f"  Exact Match : {model1_metrics['exact_match']*100:.1f}%")
print(f"  BERTScore F1: {model1_metrics['bertscore']['f1']:.4f}")
print(f"  Faithful    : {model1_metrics['faithfulness']['faithfulness_rate']*100:.1f}%")
print(f"  Hallucinate : {model1_metrics['faithfulness']['hallucination_rate']*100:.1f}%")
print("=" * 52)
print("\n➡️  Now open RAG.ipynb for Model 2!")